# M6 · RecSys (recommender system) landscape

_Curriculum · Domain 1 · Ranking & Recommenders_

**Choose the recommender family that matches the data, latency, and product question.**

We build a tiny matrix-factorization retrieval example, then treat its dot products as the first stage of a retrieval-to-ranking funnel. Run top to bottom. _Save a copy to your Drive (File -> Save a copy in Drive) to keep your edits._

In [ ]:
# Setup - CPU-only and deterministic.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(6)

## First, look at the data

Rows are users or briefs, columns are items or creators. Most production matrices are sparse; this one is small enough to see. A low-rank model predicts $\hat r_{ui}=p_u^\top q_i$.

In [ ]:
users = ["brief_A", "brief_B", "brief_C"]
items = ["creator_tech", "creator_hybrid", "creator_event"]
R = np.array([[5.0, 3.0, 0.0], [0.0, 2.0, 5.0], [4.0, 4.0, 1.0]])
ratings = pd.DataFrame(R, index=users, columns=items)

print(ratings)

## The model, in one formula

Matrix factorization approximates the interaction matrix with two smaller matrices:

$$R \approx P Q^\top$$

The retrieval score for one user and item is their dot product $p_u^\top q_i$.

### Step 1 - Factor with SVD

For a compact demonstration, truncated SVD gives two latent dimensions. Real systems learn factors with losses, sampling, and regularization.

In [ ]:
U, S, Vt = np.linalg.svd(R, full_matrices=False)
k = 2
P = U[:, :k] * np.sqrt(S[:k])
Q = Vt[:k, :].T * np.sqrt(S[:k])
R_hat = P @ Q.T

print(np.round(R_hat, 2))

assert R_hat.shape == R.shape

### Step 2 - Retrieve candidates for one brief

We score every creator by dot product and keep the top candidates. This is retrieval, not the final ranker.

In [ ]:
target = 0
scores = P[target] @ Q.T
order = np.argsort(-scores)
retrieved = [items[i] for i in order[:2]]

print(pd.Series(scores, index=items).sort_values(ascending=False))
print("top candidates:", retrieved)

assert retrieved[0] == "creator_tech"

### Step 3 - Add a lightweight ranking feature

A final ranker can mix retrieval affinity with business features. Here we add availability and compute a simple combined score.

In [ ]:
availability = np.array([0.7, 1.0, 0.4])
rank_score = 0.8 * scores + 0.2 * availability
ranked = [items[i] for i in np.argsort(-rank_score)]

print(pd.DataFrame({"retrieval": scores, "availability": availability, "rank_score": rank_score}, index=items).round(3))
print(ranked)

assert ranked[0] in retrieved

## Visualize retrieval scores

The bar chart shows why a cheap dot-product stage is so useful: it quickly separates plausible candidates from the rest.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(items, scores, color="#4c78a8")
ax.set_ylabel("dot-product score")
ax.set_title("retrieval scores for brief_A")
ax.tick_params(axis="x", rotation=20)
plt.show()

## Practice

Try each in the empty cell below it.

1. Change `k` to 1 and compare reconstruction error.
2. Add a fourth creator vector by hand and score it against `brief_A`.
3. Change the ranking weight from 0.8 to 0.5 and see whether the order changes.

In [ ]:
# Your turn:
